<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/basic_gp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian process regression with `gpc`

GPc port of [`GPy/basic_gp.ipynb`](../GPy/basic_gp.ipynb) originally written by Nicolas Durrande / James Hensman / Neil Lawrence. In GPc the modelling runs in the C++ core; this notebook is a thin NumPy / matplotlib boundary.

**Differences from GPy to watch for**

* `m.objective()` is **−log-likelihood** (SCG minimises it).
* Observation noise is `likelihood.variance` (CIP-0027); Bias/White on the kernel are opt-in.
* Parameters use `set_by_path` / `get_by_path` / `param_paths`, not `m["rbf.lengthscale"]`.
* No `optimize_restarts` / `plot_density` yet; 2-D slices are drawn with matplotlib (`predict` on a grid) rather than GPy `fixed_inputs`.
* Matérn ARD uses a shared `lengthscale` plus per-dimension `inputScale_*` (not a lengthscale vector as in GPy).


In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d


## 1-dimensional model

Toy data: noisy observations of a sine.


In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-3.0, 3.0, size=(20, 1))
Y = np.sin(X) + rng.normal(scale=0.05, size=X.shape)


Build an RBF kernel and an FTC Gaussian regression model. The shaded region in the plots is mean ± 2σ (~95% for a Gaussian).


In [ ]:
kernel = gpc.kern.RBF(1, variance=1.0, lengthscale=1.0)
m = gpc.models.GPRegression(X, Y, kernel=kernel)
print(m)
print("paths:", m.param_paths())


Predictive mean ± 2σ **before** optimisation:


In [ ]:
plot_1d(m, X, Y, label="before optimize")
plt.show()


Optimise hyperparameters (C++ SCG). Using `messages=True` allows the parameters to be monitored during optimisation.


In [ ]:
m.optimize(iters=100, messages=True)
print(m)
print("objective (−log lik):", m.objective())
plot_1d(m, X, Y, label="after optimize")
plt.show()


### Path-based parameter access

GPy: `m["rbf.lengthscale"] = 0.5`. Here we set the lengthscale with `set_by_path` and **plot before re-optimising**, so you can see the effect of a shorter correlation length (wigglier mean, tighter local structure).


In [ ]:
print("lengthscale before set:", m.get_by_path("rbf.lengthscale"))
m.set_by_path("rbf.lengthscale", 0.5)
print("after set:", m.get_by_path("rbf.lengthscale"))
print(m)
plot_1d(m, X, Y, label="lengthscale = 0.5 (not re-optimised)")
plt.show()


Re-optimise from that shortened lengthscale. SCG typically walks lengthscale back toward the earlier optimum (~1.5 on this toy sine); the set only changes the **starting** value unless you fix the parameter.


In [ ]:
m.optimize(iters=50, messages=False)
print("lengthscale after re-optimise:", m.get_by_path("rbf.lengthscale"))
print(m)
plot_1d(m, X, Y, label="after re-optimise")
plt.show()


## 2-dimensional example

Same synthetic surface as the GPy twin: \(y = \sin(x_0)\sin(x_1) + \varepsilon\), with a Matérn-5/2 ARD kernel plus a small kernel `White` (observation noise still on `likelihood.variance`).


In [ ]:
rng = np.random.default_rng(1)
X2 = rng.uniform(-3.0, 3.0, size=(50, 2))
Y2 = np.sin(X2[:, 0:1]) * np.sin(X2[:, 1:2]) + rng.normal(scale=0.05, size=(50, 1))

ker2 = gpc.kern.Matern52(2, ARD=True) + gpc.kern.White(2, variance=1e-6)
m2 = gpc.models.GPRegression(X2, Y2, kernel=ker2)
m2.set_by_path("likelihood.variance", 0.05)
print(m2)
print("paths:", m2.param_paths())
m2.optimize(iters=200, messages=True)
print(m2)
print("objective:", m2.objective())
for p in m2.param_paths():
    if "length" in p or "inputScale" in p or "Mat52.variance" in p:
        print(f"  {p} = {m2.get_by_path(p)}")


`ARD=True` makes the GP anisotropic. In `gpc` that appears as `inputScale_0` / `inputScale_1` alongside a shared `lengthscale` (effective scale per axis is modulated by those factors). The contour below shows the predictive **mean** only (GPy's 2-D `plot` does the same).


In [ ]:
xx, yy = np.meshgrid(np.linspace(-3.0, 3.0, 60), np.linspace(-3.0, 3.0, 60))
grid = np.column_stack([xx.ravel(), yy.ravel()])
mu_grid, _ = m2.predict(grid)
zz = mu_grid.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
c = ax.contourf(xx, yy, zz, levels=20)
ax.scatter(X2[:, 0], X2[:, 1], c="k", s=12, alpha=0.8)
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.set_title("2-D Matérn-ARD predictive mean")
fig.colorbar(c, ax=ax, shrink=0.85)
fig.tight_layout()
plt.show()


## Plotting slices

GPy passes `fixed_inputs=[(dim, value)]` into `m.plot`. Here we build the same slices with `predict` on a 1-D grid in the free coordinate.


In [ ]:
def plot_slices(model, Xtrain, Ytrain, fixed_dim, slice_vals, free_lim=(-3.0, 3.0), n=120):
    """fixed_dim in {0,1}; vary the other coordinate."""
    free_dim = 1 - fixed_dim
    t = np.linspace(free_lim[0], free_lim[1], n)
    fig, axes = plt.subplots(len(slice_vals), 1, figsize=(7, 2.2 * len(slice_vals)), sharex=True)
    if len(slice_vals) == 1:
        axes = [axes]
    for ax, val in zip(axes, slice_vals):
        Xstar = np.zeros((n, 2))
        Xstar[:, free_dim] = t
        Xstar[:, fixed_dim] = val
        mu, var = model.predict(Xstar)
        sd = np.sqrt(np.maximum(var, 0.0))
        ax.plot(t, mu[:, 0], "C0-")
        ax.fill_between(t, (mu - 2 * sd)[:, 0], (mu + 2 * sd)[:, 0], color="C0", alpha=0.2)
        near = np.abs(Xtrain[:, fixed_dim] - val) < 0.3
        if np.any(near):
            ax.plot(Xtrain[near, free_dim], Ytrain[near, 0], "kx", ms=4, alpha=0.7)
        ax.set_ylabel(f"y | x{fixed_dim}={val:g}")
    axes[-1].set_xlabel(f"x{free_dim}")
    fig.suptitle(f"slices with x{fixed_dim} fixed")
    fig.tight_layout()
    plt.show()

slices = [-1.0, 0.0, 1.5]
plot_slices(m2, X2, Y2, fixed_dim=1, slice_vals=slices)  # horizontal: fix x1
plot_slices(m2, X2, Y2, fixed_dim=0, slice_vals=slices)  # vertical: fix x0


Notes matching the GPy twin:

* Nearby training points are overlaid when they fall close to the slice; points with a different fixed-coordinate value are omitted, so the data can look sparse on a given row.
* Uncertainty grows away from the data, as in the 1-D plots.
